# RAG Assignment: Baseline vs. Cross-Encoder Re-Ranking

**Document:** *CATAN – Game Rules & Almanac* (5th edition, 2020 print, 16 pages), the official rulebook published by Catan GmbH / Catan Studio.
It is a good stress test for retrieval because the same concept (trading, the robber, development cards…) is described in up to three
places (short Game Rules, alphabetical Almanac, back-cover Overview), and many rules have an exception or a "you may not…" clause that sits in a
different paragraph from the general rule.

| Component | Choice |
|---|---|
| PDF parsing | `pypdf` + custom cleaning |
| Chunking | sentence-aware, ~90 words (a long sentence is never split), 1-sentence overlap, never crosses a page |
| Bi-Encoder (Pipeline 1) | `sentence-transformers/all-MiniLM-L6-v2` (384-d), cosine similarity over L2-normalised vectors |
| Vector store | in-memory NumPy matrix (exact inner-product search) |
| Cross-Encoder (Pipeline 2) | `cross-encoder/ms-marco-MiniLM-L6-v2` |
| LLM | `qwen3:8b` (Q4_K_M) served by Ollama over its HTTP `/api/chat` endpoint, thinking disabled, temperature 0. |

Both pipelines share **the same chunks, the same bi-encoder top-10 candidate list and the same prompt**. The only difference is which 3 chunks reach the LLM.

## 1. Setup

In [1]:
# pip install pypdf cryptography sentence-transformers torch pandas
import os, re, time, json, logging, warnings, urllib.request
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download progress widgets in the saved notebook
import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")
logging.getLogger("pypdf").setLevel(logging.ERROR)   # pypdf prints font warnings for this PDF
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PDF_PATH = "data/catan_base_rules_2020.pdf"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L6-v2"
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://10.1.85.86:11434/api/chat")
OLLAMA_MODEL = "qwen3:8b"

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT = 3     # chunks given to the LLM

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| LLM:", OLLAMA_MODEL, "@", OLLAMA_URL)

device: cuda | LLM: qwen3:8b @ http://10.1.85.86:11434/api/chat


## 2. Load and clean the PDF

The rulebook is a designed booklet, so raw extraction contains a lot of layout noise that would pollute the embeddings:
- `Y` glyphs that mark Almanac cross-references (e.g. `resources Y for`), and words split by them (`Y\nou` → `You`),
- small-caps headings extracted as `settin G`, `KniGht`, `lon Gest R oad`,
- number-token pips (`••••`), page numbers, `© 2015 Catan GmbH` captions, index dot-leaders,
- text printed on card illustrations (e.g. the victory-point card caption appears ~15 times).

These are removed with a few regular expressions; the rules text itself is not altered.

In [2]:
# lines that are pure layout noise: blank, bullets, pips/numbers, doubled letters ("BB"), copyright, index leaders, figure labels
NOISE_LINE = re.compile(
    r"^(\s*|[•\s]+|[\d\s•:=]+|[A-Z]{2}(\s[A-Z]{2})*|©.*|\.?(\s*\.)*\s*\d*|"
    r"Illustration [A-Z]|(Robber|Harbor|Path|Intersection){2})$"
)

# small-caps headings are extracted with stray spaces/capitals
SPLIT_HEADINGS = {
    "Constru CtinG": "constructing", "settin G": "setting", "endin G": "ending",
    "Buildin G": "building", "distan Ce": "distance", "domesti C": "domestic",
    "interse Ctions": "intersections", "resour Ce": "resource", "produ Ction": "production",
    "rollin G": "rolling", "varia Ble": "variable", "almana C": "almanac",
    "startin G": "starting", "laRG est aRmy": "largest army", "lon Gest R oad": "longest road",
    "Odds fOr dice rOlls": "odds for dice rolls",
}


def clean_page(text):
    text = re.sub(r"\bY\s*\nou\b", "You", text)          # "Y\nou" -> "You"
    lines = [l.strip() for l in text.split("\n") if not NOISE_LINE.match(l.strip())]
    text = " ".join(lines)
    text = re.sub(r"\s+Y(?=[\s.,)”:])", "", text)        # Almanac cross-reference markers
    # captions printed on card artwork (repeated many times, not rules text)
    text = re.sub(r"1 Victory Point! Reveal this card on your turn if, with it, you reach the number "
                  r"of points required for victory\.( (LIBRARY|MARKET|CHAPEL|GREAT HALL|UNIVERSITY))?", "", text)
    text = re.sub(r"2 Victory Points! (This card goes to the player with the longest road of at least 5 "
                  r"segments\. Another player who builds a longer road takes this card\.|The first player "
                  r"to play 3 [Kk]night cards gets this card\. Another player who plays more [Kk]night cards "
                  r"takes this card\.)", "", text)
    for bad, good in SPLIT_HEADINGS.items():
        text = text.replace(bad, good)
    text = re.sub(r"\b[a-z]+[A-Z][A-Za-z]*\b", lambda m: m.group(0).lower(), text)  # "KniGht" -> "knight"
    return re.sub(r"\s+", " ", text).strip()


reader = PdfReader(PDF_PATH)
pages = [(i + 1, clean_page(p.extract_text() or "")) for i, p in enumerate(reader.pages)]
pages = [(n, t) for n, t in pages if len(t.split()) > 20]   # drops the cover page
print(f"{len(reader.pages)} pages in PDF, {len(pages)} with text, {sum(len(t.split()) for _, t in pages)} words")
print("\nSample (page 5):\n", pages[3][1][:700], "...")

16 pages in PDF, 15 with text, 8541 words

Sample (page 5):
 b) Settlement Requires: Brick, Lumber, Wool, & Grain Take special note of the “Distance Rule”: you may only build a settlement at an intersection if all 3 of the adjacent intersections are vacant (i.e., none are occupied by any settlements or cities—even yours). Each of your settlements must connect to at least 1 of your own roads. Regardless of whose turn it is (i.e., during any production phase), when a terrain hex produces resources, you receive 1 resource card for each settlement you have adjacent to that terrain hex. Each settlement is worth 1 victory point. c) City Requires: 3 Ore & 2 Grain You may only establish a city by upgrading one of your settlements. When you upgrade a settlemen ...


## 3. Chunking

Sentence-aware chunks of at most ~90 words. Rules in this booklet are short, so small chunks keep one rule (or one exception) per chunk.
Each chunk repeats the previous chunk's last sentence for context (only if it is shorter than 40 words), and chunks never cross a page, so every chunk can be cited by page number.

In [3]:
def chunk_pages(pages, max_words=90, overlap_sents=1):
    chunks = []
    for page_no, text in pages:
        sents = re.split(r"(?<=[.!?])\s+(?=[A-Z(“\"])", text)
        cur = []
        for s in sents:
            if cur and sum(len(x.split()) for x in cur) + len(s.split()) > max_words:
                chunks.append({"page": page_no, "text": " ".join(cur)})
                cur = cur[-overlap_sents:] if overlap_sents and len(cur[-1].split()) < 40 else []
            cur.append(s)
        if cur and len(" ".join(cur).split()) > 8:
            chunks.append({"page": page_no, "text": " ".join(cur)})
    for i, c in enumerate(chunks):
        c["id"] = i
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
lens = [len(t.split()) for t in chunk_texts]
print(f"{len(chunks)} chunks | words per chunk: mean {np.mean(lens):.0f}, min {min(lens)}, max {max(lens)}")
pd.DataFrame(chunks)[["id", "page", "text"]].head(8)

133 chunks | words per chunk: mean 78, min 32, max 217


,id,page,text
0,0,2,"Dear Settlers, To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system,..."
1,1,2,Now you are ready for your first adventure on Catan. Have fun settling this new land together! — Klaus Teuber You ca...
2,2,2,"If you need more information during the game, you can look up keywords (marked) in the “Almanac,” which follows thes..."
3,3,2,"Before building the island, assemble the frame by matching the numbers at the ends of the frame pieces together (i.e..."
4,4,2,"A on page 3.) This set-up is well-balanced for all players. Before your first game, you must remove the die-cut comp..."
5,5,2,"First, assemble the frame as shown. Second, create Catan by placing the 19 terrain hexes on the table—again as shown..."
6,6,3,Starting Set-up for Experienced Players It is more fun to play with a variable game board—with the game board laid o...
7,7,3,Also look for useful tips under Set-up Phase and Tactics. setting up the Game Select a color and take your 5 settlem...


## 4. Pipeline 1 – Bi-Encoder index and baseline retrieval

Every chunk is embedded **once, offline** by the bi-encoder. Vectors are L2-normalised so the inner product equals cosine similarity.
At query time only the query is embedded, and a single matrix–vector product scores it against all chunks.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True)
index_time = time.perf_counter() - t0
print(f"Indexed {chunk_emb.shape[0]} chunks -> matrix {chunk_emb.shape} in {index_time:.2f}s")


def retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1, step b: Top-k chunks by cosine similarity
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    sims = chunk_emb @ q
    top = np.argsort(-sims)[:k]
    return [{"id": int(i), "page": chunks[i]["page"], "bi_rank": r + 1, "bi_score": float(sims[i]),
             "text": chunks[i]["text"]} for r, i in enumerate(top)]


def full_bi_rank(query, chunk_id):
    # rank of a chunk over the WHOLE corpus (used to diagnose recall misses outside the top 10)
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    return int(np.where(np.argsort(-(chunk_emb @ q)) == chunk_id)[0][0]) + 1

Indexed 133 chunks -> matrix (133, 384) in 0.31s


## 5. Pipeline 2 – Cross-Encoder re-ranking

The cross-encoder reads `[CLS] query [SEP] chunk [SEP]` as **one** sequence, so every query token can attend to every chunk token.
It outputs a single relevance logit per pair. It is applied **only to the 10 bi-encoder candidates**.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)


def rerank(query, candidates):
    # Pipeline 2, steps b–c: score all (query, chunk) pairs, sort descending
    scores = cross_encoder.predict([(query, c["text"]) for c in candidates], batch_size=16)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]
    out.sort(key=lambda c: -c["ce_score"])
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

## 6. LLM and the shared prompt

The **same** prompt template is used for Answer 1 (bi-encoder top-3) and Answer 2 (re-ranked top-3). Decoding is greedy (temperature 0), so any
difference between the two answers comes from the context and not from sampling.

In [6]:
PROMPT_TEMPLATE = """You are a rules assistant for the board game CATAN.
Answer the question using ONLY the rulebook excerpts below. If the excerpts do not contain the answer, say "The excerpts do not say."
Answer in at most 3 sentences and mention any exception that applies.

Rulebook excerpts:
{context}

Question: {question}
Answer:"""


def build_prompt(question, ctx_chunks):
    context = "\n\n".join(f"[Excerpt {i+1} | page {c['page']}]\n{c['text']}" for i, c in enumerate(ctx_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)


def generate(prompt, max_new_tokens=150):
    # same request as: POST {OLLAMA_URL} {"model": "qwen3:8b", "messages": [...], "stream": false}
    body = {"model": OLLAMA_MODEL, "messages": [{"role": "user", "content": prompt}], "stream": False,
            "think": False,   # qwen3 is a reasoning model; disable <think> so the answer is direct
            "options": {"temperature": 0, "num_predict": max_new_tokens}}
    req = urllib.request.Request(OLLAMA_URL, data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["message"]["content"].strip()


print(generate("Reply with the single word: ready"))


ready


## 7. The two end-to-end pipelines (with timing)

`time.perf_counter()` is recorded around each stage. Retrieval is shared, so Pipeline 2's retrieval cost is the baseline's cost plus the re-ranking cost.

In [7]:
def run_both(query):
    t0 = time.perf_counter(); cands = retrieve(query); t_retrieve = time.perf_counter() - t0
    t0 = time.perf_counter(); reranked = rerank(query, cands); t_rerank = time.perf_counter() - t0

    top3_base, top3_rr = cands[:TOP_K_CONTEXT], reranked[:TOP_K_CONTEXT]
    t0 = time.perf_counter(); ans1 = generate(build_prompt(query, top3_base)); t_llm1 = time.perf_counter() - t0
    t0 = time.perf_counter(); ans2 = generate(build_prompt(query, top3_rr)); t_llm2 = time.perf_counter() - t0
    return {"query": query, "candidates": cands, "reranked": reranked, "answer1": ans1, "answer2": ans2,
            "t_retrieve": t_retrieve, "t_rerank": t_rerank, "t_llm1": t_llm1, "t_llm2": t_llm2}


def show(res, gold_phrases):
    # top-10 table: bi-encoder rank vs cross-encoder rank; * marks the chunk that actually answers the question
    rows = []
    for c in res["reranked"]:
        gold = any(g in c["text"].lower() for g in gold_phrases)
        rows.append({"chunk": c["id"], "page": c["page"], "bi_rank": c["bi_rank"], "bi_cos": round(c["bi_score"], 3),
                     "ce_rank": c["ce_rank"], "ce_score": round(c["ce_score"], 2), "gold": "*" if gold else "",
                     "text": c["text"][:110] + "…"})
    df = pd.DataFrame(rows).sort_values("bi_rank").set_index("bi_rank")
    print("QUERY:", res["query"])
    display(df)
    print("Baseline top-3 chunks:", [c["id"] for c in res["candidates"][:3]],
          "| Re-ranked top-3 chunks:", [c["id"] for c in res["reranked"][:3]])
    print("\nANSWER 1 (baseline):\n ", res["answer1"])
    print("\nANSWER 2 (re-ranked):\n ", res["answer2"])

## 8. Failure-mode stress tests

For every query a *gold* phrase identifies the chunk(s) that contain the deciding rule, so rank shifts are measured objectively rather than by eye.
Queries 1–5 are cases where the baseline misses or demotes the key chunk and re-ranking recovers it. Queries 6–7 are included on purpose
to show where re-ranking does **not** help.

In [8]:
FAILURE_QUERIES = [
    {"type": "Negation & constraints",
     "query": "Is it prohibited to give resource cards to another player for free?",
     "gold": ["you may not give away cards", "cannot give away cards"]},
    {"type": "Exception / corner case",
     "query": "Which development card can I reveal on the same turn I buy it?",
     "gold": ["exception: if you buy a card"]},
    {"type": "Keyword trap / distractor",
     "query": "Can a settlement be built right next to my own settlement?",
     "gold": ["even yours"]},
    {"type": "Exception (rule variant)",
     "query": "Can I use a harbor on the same turn I build a settlement on it?",
     "gold": ["use a harbor on the same turn"]},
    {"type": "Buried chunk (ranks 4–10)",
     "query": "After playing Year of Plenty, can I spend those cards to build right away?",
     "gold": ["use these cards to build in the same turn"]},
    {"type": "Counter-example: re-ranker demotes the answer",
     "query": "Can I trade 2 wool for 1 wool?",
     "gold": ["trade like resources", "trade matching resources"]},
    {"type": "Counter-example: recall miss (not in top-10)",
     "query": "Am I allowed to leave the robber where it is after rolling a 7?",
     "gold": ["may not choose to leave the robber"]},
]

# warm-up so that one-off model initialisation is not counted in the latency numbers
_ = run_both("How many victory points do I need to win?")

results = []
for fq in FAILURE_QUERIES:
    results.append(run_both(fq["query"]))

In [9]:
print("TYPE:", FAILURE_QUERIES[0]["type"])
show(results[0], FAILURE_QUERIES[0]["gold"])

TYPE: Negation & constraints
QUERY: Is it prohibited to give resource cards to another player for free?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,75,10,0.631,5,0.29,,"If an opponent does not have a resource card of the specified type, they do not have to give you anything. You…"
2,74,10,0.568,3,2.52,,"Monopoly: If you play this card, you must name 1 type of resource. All the other players must give you all of …"
3,16,4,0.558,4,1.71,,"If there are not enough of a given resource in the supply to fulfill everyone’s production, then no one receiv…"
4,81,10,0.523,8,-0.67,,Take any 2 resources from the bank. Add them to your hand. They can be 2 of the same resource or 2 different r…
5,80,10,0.511,2,2.65,,"If there are not enough resource cards to give every player all the production they earn, then no player recei…"
6,17,4,0.482,6,-0.45,,You can announce which resources you need and what you are willing to trade for them. The other players can al…
7,79,10,0.430,9,-1.12,,Bridget takes 1 brick card and 1 wool card from the supply stacks. It is possible that during the game there w…
8,67,9,0.401,10,-7.59,,"He can, for instance, exchange 3 lumber cards for 1 wool card. Special Harbor (2:1): There is only 1 special h…"
9,86,11,0.391,7,-0.49,,This lasts until the robber is moved again by another “7” or a knight card. Xander may also take 1 resource ca…


Baseline top-3 chunks: [75, 74, 16] | Re-ranked top-3 chunks: [107, 80, 74]

ANSWER 1 (baseline):
  The excerpts do not say. According to the rules, players may trade resource cards freely with other players during their turn, which includes giving resource cards to another player for free as part of a trade. There is no prohibition mentioned in the provided excerpts.

ANSWER 2 (re-ranked):
  No, it is not prohibited to give resource cards to another player for free, except during another player's turn. You may trade with another player between your turns, but only if it is that player’s turn and they elect to trade with you. You may not give away cards during your turn if you decide not to trade.


In [10]:
print("TYPE:", FAILURE_QUERIES[1]["type"])
show(results[1], FAILURE_QUERIES[1]["gold"])

TYPE: Exception / corner case
QUERY: Which development card can I reveal on the same turn I buy it?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,45,7,0.586,1,7.18,,Keep your development cards hidden until you play them. This keeps the other players in the dark. You cannot t…
2,25,5,0.503,4,2.46,,"There are 3 different kinds of these cards: knight, progress, and victory point. Each has a different effect (…"
3,53,8,0.440,8,-3.31,,"Game play Here is a summary of the the game sequence, plus some more specific entries where you can find detai…"
4,46,7,0.423,2,5.71,*,"You may not, however, play a card that you bought during the same turn. Exception: If you buy a card and it is…"
5,13,4,0.412,3,3.09,,"You may trade resource cards with other players and/or use maritime trade. You may build roads, settlements or…"
6,19,4,0.399,6,-1.23,,"Build Now you can build. Through building, you can increase your victory points, expand your road network, imp…"
7,72,10,0.387,9,-3.78,,Paths run along the border of 2 terrain hexes or between a land hex and the frame. Only one road can be built …
8,24,5,0.381,7,-1.62,,"When you upgrade a settlement to a city, put the settlement (house) piece back in your supply and replace it w…"
9,116,15,0.381,10,-5.82,,Combined Trade/Build Phase Desert. . . . . . . . . . . . . . . . . . . . . . . . . . . . Development Cards Dis…


Baseline top-3 chunks: [45, 25, 53] | Re-ranked top-3 chunks: [45, 46, 13]

ANSWER 1 (baseline):
  You cannot reveal a development card on the same turn you buy it. The rule states that you may not play a card that you bought during the same turn.

ANSWER 2 (re-ranked):
  The development card you can reveal on the same turn you buy it is a victory point card that brings you to 10 points. This is an exception to the rule that you may not play a card that you bought during the same turn. You may immediately reveal this card (and all other VP cards) and win the game.


In [11]:
print("TYPE:", FAILURE_QUERIES[2]["type"])
show(results[2], FAILURE_QUERIES[2]["gold"])

TYPE: Keyword trap / distractor
QUERY: Can a settlement be built right next to my own settlement?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,96,12,0.568,10,-0.25,,"Important: When placing all other settlements, the Distance Rule (see page 7) always applies! Round Two Once a…"
2,97,12,0.557,9,-0.16,,"The second settlement can be placed on any unoccupied intersection, as long as the Distance Rule is observed. …"
3,39,6,0.548,5,2.58,,"You can build settlements and roads, upgrade settlements to cities, and buy development cards. Cities You cann…"
4,40,6,0.533,6,2.54,,"When you build a city, the upgraded settlement piece becomes available again. You can use it to build another …"
5,22,5,0.531,2,3.79,*,"b) Settlement Requires: Brick, Lumber, Wool, & Grain Take special note of the “Distance Rule”: you may only bu…"
6,93,12,0.529,1,3.88,,"Note: If you have built all 5 of your settlements, you must upgrade 1 of your settlements to a city before you…"
7,105,13,0.507,8,0.33,,"Do not underestimate the value of harbors. For instance, a player with settlements or cities on productive fie…"
8,128,16,0.506,4,2.84,,A successful trade might yield you a big build! You can only build a new settlement on an unoccupied intersect…
9,90,11,0.506,7,2.48,,You share in all of the resource production of each terrain hex adjacent to your settlements. You must meet 2 …


Baseline top-3 chunks: [96, 97, 39] | Re-ranked top-3 chunks: [93, 22, 92]

ANSWER 1 (baseline):
  The excerpts do not say. The Distance Rule (see page 7) must be observed when placing settlements, which implies that a settlement cannot be built directly next to another settlement. However, the second settlement does not need to connect to the first settlement.

ANSWER 2 (re-ranked):
  No, a settlement cannot be built right next to your own settlement. According to the Distance Rule, you may only build a settlement at an intersection if all 3 of the adjacent intersections are vacant. This means you cannot build a settlement next to your own settlement.


In [12]:
print("TYPE:", FAILURE_QUERIES[3]["type"])
show(results[3], FAILURE_QUERIES[3]["gold"])

TYPE: Exception (rule variant)
QUERY: Can I use a harbor on the same turn I build a settlement on it?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,105,13,0.714,5,4.02,,"Do not underestimate the value of harbors. For instance, a player with settlements or cities on productive fie…"
2,54,8,0.660,3,5.94,,"On your turn, you complete these 3 phases in order: • Roll for Resource Production (the roll applies to all pl…"
3,106,13,0.655,6,3.98,,"If you plan to build toward a harbor , the middle of the island may be a tricky place for a starting settlemen…"
4,42,6,0.573,7,3.06,,"You can build settlements and upgrade settlements to cities on intersections that border on the sea. However, …"
5,43,6,0.559,1,8.74,*,"We recommend experienced players ignore this separation. After rolling for resource production, you can trade …"
6,55,8,0.532,4,5.12,,"In order to control a harbor, you must build a settlement on a coastal intersection which borders the harbor. …"
7,18,4,0.461,2,6.04,,"During your turn, you can always trade at 4:1 by putting 4 identical resource cards back in their stack and ta…"
8,100,13,0.457,10,-8.14,,2. Now take the 9 harbor pieces (the small 5-sided pieces with ships on them) and randomly place one on top of…
9,67,9,0.450,8,-0.65,,"He can, for instance, exchange 3 lumber cards for 1 wool card. Special Harbor (2:1): There is only 1 special h…"


Baseline top-3 chunks: [105, 54, 106] | Re-ranked top-3 chunks: [43, 18, 54]

ANSWER 1 (baseline):
  No, you cannot use a harbor on the same turn you build a settlement on it. Building a settlement on a harbor is part of the "Build" phase, and using the harbor for trading is part of the "Trade" phase, which occurs after the "Build" phase. The excerpts do not mention any exception for this rule.

ANSWER 2 (re-ranked):
  Yes, you can use a harbor on the same turn you build a settlement on it. The excerpts indicate that you can trade and build in any order during your turn, and you can even use a harbor on the same turn you build a settlement there. There is no exception mentioned for this scenario.


In [13]:
print("TYPE:", FAILURE_QUERIES[4]["type"])
show(results[4], FAILURE_QUERIES[4]["gold"])

TYPE: Buried chunk (ranks 4–10)
QUERY: After playing Year of Plenty, can I spend those cards to build right away?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,37,6,0.575,3,-3.32,,You can build as many items and buy as many cards as you desire—as long as you have enough resources to “pay” …
2,36,6,0.548,7,-5.66,,Build (building) You may build on your turn after you have rolled for resource production and finished trading…
3,45,7,0.514,6,-5.58,,Keep your development cards hidden until you play them. This keeps the other players in the dark. You cannot t…
4,19,4,0.499,4,-5.13,,"Build Now you can build. Through building, you can increase your victory points, expand your road network, imp…"
5,73,10,0.493,1,2.95,*,"There are 2 each of 3 varieties: Road Building: If you play this card, you may immediately place 2 free roads …"
6,81,10,0.491,2,0.70,,Take any 2 resources from the bank. Add them to your hand. They can be 2 of the same resource or 2 different r…
7,13,4,0.462,5,-5.39,,"You may trade resource cards with other players and/or use maritime trade. You may build roads, settlements or…"
8,108,14,0.445,9,-8.47,,"You may not give away cards. You may not trade development cards. You may not trade like resources (e.g., 2 wo…"
9,75,10,0.444,10,-9.53,,"If an opponent does not have a resource card of the specified type, they do not have to give you anything. You…"


Baseline top-3 chunks: [37, 36, 45] | Re-ranked top-3 chunks: [73, 81, 37]

ANSWER 1 (baseline):
  The excerpts do not say. You can build as many items as you desire as long as you have enough resources and they are still available in the supply, but the rulebook does not specify whether you can immediately use the resource cards from Year of Plenty to build.

ANSWER 2 (re-ranked):
  Yes, after playing the Year of Plenty card, you can spend those cards to build right away. The rulebook excerpts state that you may use these cards to build in the same turn. There is no exception mentioned for this specific scenario.


In [14]:
print("TYPE:", FAILURE_QUERIES[5]["type"])
show(results[5], FAILURE_QUERIES[5]["gold"])

TYPE: Counter-example: re-ranker demotes the answer
QUERY: Can I trade 2 wool for 1 wool?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,69,9,0.621,1,2.72,,"He can also trade 4 ore cards for any 2 other cards. If he traded 4 wool instead of 4 ore, he would get only 1…"
2,67,9,0.488,2,1.25,,"He can, for instance, exchange 3 lumber cards for 1 wool card. Special Harbor (2:1): There is only 1 special h…"
3,49,7,0.450,5,-5.79,*,"You may trade as many times as you can, using single or multiple cards. However, you cannot give away cards, o…"
4,127,16,0.438,4,-0.51,,"Finally, a settlement at [E] would only harvest the production from 1 terrain hex (pasture). However [E] is al…"
5,68,9,0.372,9,-9.40,,The exchange rate of 2:1 only applies to the resource shown on the harbor location. A special harbor does not …
6,50,7,0.365,6,-7.99,,"He has 2 lumber and 3 ore. Pete asks aloud, “Who will give me 1 brick for 1 ore?” Beth answers, “If you give m…"
7,17,4,0.362,8,-8.97,,You can announce which resources you need and what you are willing to trade for them. The other players can al…
8,66,9,0.351,3,0.11,,"Normally, he should first try a more favorable trade with the other players (domestic trade). If you have buil…"
9,65,9,0.328,7,-8.90,,The most basic (and unfavorable) exchange rate is 4:1. You may trade 4 identical resource cards to the supply …


Baseline top-3 chunks: [69, 67, 49] | Re-ranked top-3 chunks: [69, 67, 66]

ANSWER 1 (baseline):
  The excerpts do not say. You cannot trade matching resources, such as 2 wool for 1 wool, as it is considered a non-allowed trade.

ANSWER 2 (re-ranked):
  The excerpts do not say. There is no rule mentioned that allows trading 2 wool for 1 wool. The rulebook excerpts only describe trade ratios like 2:1 or 3:1, not 1:1 trades.


In [15]:
print("TYPE:", FAILURE_QUERIES[6]["type"])
show(results[6], FAILURE_QUERIES[6]["gold"])

TYPE: Counter-example: recall miss (not in top-10)
QUERY: Am I allowed to leave the robber where it is after rolling a 7?


,chunk,page,bi_cos,ce_rank,ce_score,gold,text
bi_rank,,,,,,,
1,29,5,0.590,5,2.32,,"The robber prevents it. b) Playing Development Cards At any time during your turn, you may play 1 development …"
2,26,5,0.576,1,5.38,,"Special Cases a) Rolling a “7” and Activating the Robber If you roll a “7,” no one receives any resources. Ins…"
3,88,11,0.542,3,3.76,,(2) Then you (the player who rolled the “7”) must move the robber to the number token of any other terrain hex…
4,86,11,0.492,2,3.89,,This lasts until the robber is moved again by another “7” or a knight card. Xander may also take 1 resource ca…
5,33,5,0.463,8,-7.16,,"Of course, you can reveal them after the end of the game if someone else wins. You may play any number of vict…"
6,87,11,0.459,6,0.31,,"Return discards to the supply stacks. If you hold an odd number of cards, round down (e.g., if you have 9 reso…"
7,84,11,0.448,4,2.84,,"Each of these paths connects to either Liam’s road or his settlement, and is not blocked by the blue player’s …"
8,91,11,0.439,7,-3.17,,The robber was in a fields hex. Xander places it on the “4” number token of a hills hex. If a “4” is rolled in…
9,130,16,0.435,10,-9.94,,You should consider settling on hexes that have good potential for production (i.e. 6 and 8 vs. 2 and 12). How…


Baseline top-3 chunks: [29, 26, 88] | Re-ranked top-3 chunks: [26, 86, 88]

ANSWER 1 (baseline):
  No, you are not allowed to leave the robber where it is after rolling a 7. According to Excerpt 2, you must move the robber immediately to the number token of any other terrain hex or to the desert hex. There is no exception mentioned for this rule.

ANSWER 2 (re-ranked):
  No, you are not allowed to leave the robber where it is after rolling a 7. You must move the robber to the number token of any other terrain hex or to the desert hex. This is required as part of the process when activating the robber.


## 9. Rank-shift summary

In [16]:
def gold_rank(lst, gold, key):
    ranks = [c[key] for c in lst if any(g in c["text"].lower() for g in gold)]
    return min(ranks) if ranks else None

summary = []
for fq, res in zip(FAILURE_QUERIES, results):
    gold_ids = [c["id"] for c in chunks if any(g in c["text"].lower() for g in fq["gold"])]
    b = gold_rank(res["candidates"], fq["gold"], "bi_rank")
    r = gold_rank(res["reranked"], fq["gold"], "ce_rank")
    best_id = min(gold_ids, key=lambda i: full_bi_rank(fq["query"], i))
    summary.append({
        "type": fq["type"], "query": fq["query"], "gold chunk": best_id,
        "bi-encoder rank (full corpus)": full_bi_rank(fq["query"], best_id),
        "rank after re-ranking": r if r else "not in top-10",
        "in top-3: baseline": bool(b and b <= 3), "in top-3: re-ranked": bool(r and r <= 3),
    })
summary_df = pd.DataFrame(summary)
os.makedirs("results", exist_ok=True)
summary_df.to_csv("results/rank_shifts.csv", index=False)
summary_df

,type,query,gold chunk,bi-encoder rank (full corpus),rank after re-ranking,in top-3: baseline,in top-3: re-ranked
0,Negation & constraints,Is it prohibited to give resource cards to another player for free?,107,10,1,False,True
1,Exception / corner case,Which development card can I reveal on the same turn I buy it?,46,4,2,False,True
2,Keyword trap / distractor,Can a settlement be built right next to my own settlement?,22,5,2,False,True
3,Exception (rule variant),Can I use a harbor on the same turn I build a settlement on it?,43,5,1,False,True
4,Buried chunk (ranks 4–10),"After playing Year of Plenty, can I spend those cards to build right away?",73,5,1,False,True
5,Counter-example: re-ranker demotes the answer,Can I trade 2 wool for 1 wool?,49,3,5,True,False
6,Counter-example: recall miss (not in top-10),Am I allowed to leave the robber where it is after rolling a 7?,89,17,not in top-10,False,False


In [17]:
# side-by-side answers for the report
answers_df = pd.DataFrame([{"type": fq["type"], "query": r["query"], "answer 1 (baseline)": r["answer1"],
                            "answer 2 (re-ranked)": r["answer2"]} for fq, r in zip(FAILURE_QUERIES, results)])
answers_df.to_csv("results/answers.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(answers_df)

,type,query,answer 1 (baseline),answer 2 (re-ranked)
0,Negation & constraints,Is it prohibited to give resource cards to another player for free?,"The excerpts do not say. According to the rules, players may trade resource cards freely with other players during their turn, which includes giving resource cards to another player for free as part of a trade. There is no prohibition mentioned in the provided excerpts.","No, it is not prohibited to give resource cards to another player for free, except during another player's turn. You may trade with another player between your turns, but only if it is that player’s turn and they elect to trade with you. You may not give away cards during your turn if you decide not to trade."
1,Exception / corner case,Which development card can I reveal on the same turn I buy it?,You cannot reveal a development card on the same turn you buy it. The rule states that you may not play a card that you bought during the same turn.,The development card you can reveal on the same turn you buy it is a victory point card that brings you to 10 points. This is an exception to the rule that you may not play a card that you bought during the same turn. You may immediately reveal this card (and all other VP cards) and win the game.
2,Keyword trap / distractor,Can a settlement be built right next to my own settlement?,"The excerpts do not say. The Distance Rule (see page 7) must be observed when placing settlements, which implies that a settlement cannot be built directly next to another settlement. However, the second settlement does not need to connect to the first settlement.","No, a settlement cannot be built right next to your own settlement. According to the Distance Rule, you may only build a settlement at an intersection if all 3 of the adjacent intersections are vacant. This means you cannot build a settlement next to your own settlement."
3,Exception (rule variant),Can I use a harbor on the same turn I build a settlement on it?,"No, you cannot use a harbor on the same turn you build a settlement on it. Building a settlement on a harbor is part of the ""Build"" phase, and using the harbor for trading is part of the ""Trade"" phase, which occurs after the ""Build"" phase. The excerpts do not mention any exception for this rule.","Yes, you can use a harbor on the same turn you build a settlement on it. The excerpts indicate that you can trade and build in any order during your turn, and you can even use a harbor on the same turn you build a settlement there. There is no exception mentioned for this scenario."
4,Buried chunk (ranks 4–10),"After playing Year of Plenty, can I spend those cards to build right away?","The excerpts do not say. You can build as many items as you desire as long as you have enough resources and they are still available in the supply, but the rulebook does not specify whether you can immediately use the resource cards from Year of Plenty to build.","Yes, after playing the Year of Plenty card, you can spend those cards to build right away. The rulebook excerpts state that you may use these cards to build in the same turn. There is no exception mentioned for this specific scenario."
5,Counter-example: re-ranker demotes the answer,Can I trade 2 wool for 1 wool?,"The excerpts do not say. You cannot trade matching resources, such as 2 wool for 1 wool, as it is considered a non-allowed trade.","The excerpts do not say. There is no rule mentioned that allows trading 2 wool for 1 wool. The rulebook excerpts only describe trade ratios like 2:1 or 3:1, not 1:1 trades."
6,Counter-example: recall miss (not in top-10),Am I allowed to leave the robber where it is after rolling a 7?,"No, you are not allowed to leave the robber where it is after rolling a 7. According to Excerpt 2, you must move the robber immediately to the number token of any other terrain hex or to the desert hex. There is no exception mentioned for this rule.","No, you are not allowed to leave the robber where it is after roll

## 10. Latency profiling

Timings over a larger set of queries (the 7 stress tests plus 13 ordinary rules questions), after the warm-up.
LLM generation is timed separately because it is identical in cost for both pipelines (same prompt length, 3 chunks).

In [18]:
EXTRA_QUERIES = [
    "How many victory points do I need to win?", "What does it cost to build a city?",
    "How many resource cards must I discard if I have 9 cards when a 7 is rolled?",
    "Who gets the Largest Army card?", "What is the distance rule?", "How does maritime trade work?",
    "Can I trade with the bank when it is not my turn?", "What does the Monopoly card do?",
    "How many roads, settlements and cities does each player have?", "Where does the robber start the game?",
    "How is the starting player chosen?", "What happens when the bank runs out of a resource?",
    "How many development cards can I play per turn?",
]
lat_queries = [fq["query"] for fq in FAILURE_QUERIES] + EXTRA_QUERIES

N_REPEAT = 5   # retrieval stages are fast, so repeat them for stable numbers
rows = []
for q in lat_queries:
    for _ in range(N_REPEAT):
        t0 = time.perf_counter(); cands = retrieve(q); t1 = time.perf_counter()
        rerank(q, cands); t2 = time.perf_counter()
        rows.append({"query": q, "bi_retrieve_ms": (t1 - t0) * 1e3, "ce_rerank_ms": (t2 - t1) * 1e3})
lat = pd.DataFrame(rows)

llm_times = [r["t_llm1"] for r in results] + [r["t_llm2"] for r in results]
stats = lambda s: {"mean": s.mean(), "median": s.median(), "p95": s.quantile(0.95)}
llm_ms = pd.Series(llm_times) * 1e3
latency_table = pd.DataFrame({
    "Pipeline 1 retrieval (embed query + top-10 search)": stats(lat.bi_retrieve_ms),
    "Cross-encoder re-rank of 10 pairs": stats(lat.ce_rerank_ms),
    "Pipeline 2 retrieval total (bi-encoder + re-rank)": stats(lat.bi_retrieve_ms + lat.ce_rerank_ms),
    "LLM generation (one answer, either pipeline)": stats(llm_ms),
}).T.round(1)
latency_table.columns = ["mean (ms)", "median (ms)", "p95 (ms)"]
latency_table.to_csv("results/latency.csv")
print(f"{len(lat_queries)} queries x {N_REPEAT} repeats on device={DEVICE}; one-off index build: {index_time*1e3:.0f} ms for {len(chunks)} chunks")
latency_table

20 queries x 5 repeats on device=cuda; one-off index build: 310 ms for 133 chunks


,mean (ms),median (ms),p95 (ms)
Pipeline 1 retrieval (embed query + top-10 search),1.8,1.8,1.9
Cross-encoder re-rank of 10 pairs,8.7,8.4,11.5
Pipeline 2 retrieval total (bi-encoder + re-rank),10.5,10.3,13.3
"LLM generation (one answer, either pipeline)",896.9,844.7,1230.1


In [19]:
p1 = latency_table.loc["Pipeline 1 retrieval (embed query + top-10 search)", "mean (ms)"]
p2 = latency_table.loc["Pipeline 2 retrieval total (bi-encoder + re-rank)", "mean (ms)"]
llm_mean = latency_table.loc["LLM generation (one answer, either pipeline)", "mean (ms)"]
print(f"Retrieval only : re-ranking makes retrieval {p2/p1:.1f}x slower ({p1:.1f} ms -> {p2:.1f} ms)")
print(f"End-to-end     : {p1+llm_mean:.0f} ms vs {p2+llm_mean:.0f} ms  (+{100*(p2-p1)/(p1+llm_mean):.1f}%)")

Retrieval only : re-ranking makes retrieval 5.8x slower (1.8 ms -> 10.5 ms)
End-to-end     : 899 ms vs 907 ms  (+1.0%)


## 11. Scaling experiment: why not run the cross-encoder over everything?

To make the scaling argument concrete, the cross-encoder is run over **all** chunks for one query and compared with the bi-encoder,
then both are extrapolated to larger collections. The bi-encoder's per-query cost is one query encoding plus a dot product per chunk
(chunk vectors are pre-computed). The cross-encoder needs a full transformer forward pass per (query, chunk) pair, and nothing can be pre-computed because each pass depends on the query.

In [20]:
q = FAILURE_QUERIES[6]["query"]   # the recall-miss query
times_full = []
for _ in range(3):
    t0 = time.perf_counter()
    full_scores = cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=32)
    times_full.append(time.perf_counter() - t0)
ce_full_ms = np.median(times_full) * 1e3
per_pair_ms = ce_full_ms / len(chunks)

times_bi = []
for _ in range(20):
    t0 = time.perf_counter(); retrieve(q); times_bi.append(time.perf_counter() - t0)
bi_ms = np.median(times_bi) * 1e3

order = np.argsort(-full_scores)
gold_id = [c["id"] for c in chunks if "may not choose to leave the robber" in c["text"].lower()][0]
print(f"Cross-encoder over all {len(chunks)} chunks: {ce_full_ms:.0f} ms ({per_pair_ms:.2f} ms/pair) vs bi-encoder retrieval {bi_ms:.1f} ms")
print(f"Gold chunk {gold_id}: bi-encoder rank {full_bi_rank(q, gold_id)} -> full-corpus cross-encoder rank {int(np.where(order == gold_id)[0][0]) + 1}")

scale = pd.DataFrame({"chunks in collection": [len(chunks), 10_000, 1_000_000, 100_000_000]})
scale["cross-encoder over all chunks"] = (scale["chunks in collection"] * per_pair_ms / 1e3).map(
    lambda s: f"{s:,.1f} s" if s < 3600 else f"{s/3600:,.1f} h")
scale["bi-encoder top-10 + re-rank 10"] = f"~{bi_ms + latency_table.loc['Cross-encoder re-rank of 10 pairs', 'median (ms)']:.0f} ms (+ ANN index lookup)"
scale.to_csv("results/scaling.csv", index=False)
scale

Cross-encoder over all 133 chunks: 96 ms (0.72 ms/pair) vs bi-encoder retrieval 1.8 ms
Gold chunk 89: bi-encoder rank 17 -> full-corpus cross-encoder rank 11


,chunks in collection,cross-encoder over all chunks,bi-encoder top-10 + re-rank 10
0,133,0.1 s,~10 ms (+ ANN index lookup)
1,10000,7.2 s,~10 ms (+ ANN index lookup)
2,1000000,724.7 s,~10 ms (+ ANN index lookup)
3,100000000,20.1 h,~10 ms (+ ANN index lookup)


## 12. Observations (raw notes for the report)

*These are short factual notes taken from the outputs above. The written report is my own text.*

- See `results/rank_shifts.csv`, `results/answers.csv`, `results/latency.csv` and `results/scaling.csv` for the numbers used in the report.
- Queries 1–5: the deciding chunk was outside the baseline top-3 and the cross-encoder moved it into the top-3.
- Query 6: the cross-encoder pushed the correct chunk *out* of the top-3. Re-ranking is not always better.
- Query 7: the answer chunk is not in the bi-encoder top-10, so a re-ranker over 10 candidates cannot recover it. Recall is capped by the first stage.